# 06 · Modelo Late Fusion

**Objetivo:** Entrenar dos modelos independientes (uno para imágenes, otro para tabular) y combinar sus predicciones mediante un **clasificador aprendido** (SVM con kernel RBF).

**Datos de entrada:** `../data/raw/hnmist_28_28_RGB.csv`, `../data/raw/HAM10000_metadata.csv`

**Resultado esperado:** Modelos guardados en `../models/late_fusion_image_model.h5` y `../models/late_fusion_tabular_model.h5` con ~75% de accuracy combinado.

**Por qué Late Fusion es inferior a Early Fusion:** cada rama aprende de forma aislada, sin poder capturar las interacciones entre imagen y metadatos. La fusión en predicciones es una combinación de opiniones independientes, no un aprendizaje conjunto.

**Diferencia con un promedio fijo:** en lugar de usar pesos arbitrarios (`0.6 * img + 0.4 * tab`), entrenamos un SVM que **aprende** cuánto peso dar a cada modelo según los datos.

## Carga de datos, preprocesado y entrenamiento

**Estrategia de fusión:** las probabilidades de ambos modelos se concatenan (14 valores: 7 del modelo de imagen + 7 del tabular) y se usan como entrada de un SVM con kernel RBF. El SVM aprende la combinación óptima durante el entrenamiento.

```
pred_img  (n, 7)  ──┐
                     ├─→ concat (n, 14) ─→ SVM ─→ diagnóstico final
pred_tab  (n, 7)  ──┘
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ── Carga y preprocesado centralizados en utils.py ────────────────────────────
from utils import get_all_splits

(X_img_train, X_img_val, X_img_test,
 X_tab_train, X_tab_val, X_tab_test,
 y_train, y_val, y_test, le) = get_all_splits()

num_classes = y_train.shape[1]
n_tab_features = X_tab_train.shape[1]

# ── Modelo tabular ────────────────────────────────────────────────────────────
model_tab = Sequential([
    Dense(32, activation='relu', input_shape=(n_tab_features,)),
    Dropout(0.3),
    Dense(8, activation='relu'),
    Dense(num_classes, activation='softmax')
])
model_tab.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_tab.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=10, min_delta=0.005, restore_best_weights=True)
history_tab = model_tab.fit(
    X_tab_train, y_train,
    validation_data=(X_tab_val, y_val),
    epochs=100, batch_size=32,
    callbacks=[early_stop], verbose=1
)

plt.figure(figsize=(8, 5))
plt.plot(history_tab.history['loss'], label='Training loss')
plt.plot(history_tab.history['val_loss'], label='Validation loss')
plt.title('Loss — Modelo Tabular (Late Fusion)')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True); plt.show()

# ── Modelo de imágenes ────────────────────────────────────────────────────────
model_img = Sequential([
    Conv2D(32,  (3,3), activation='relu', padding='same', input_shape=(28,28,3)),
    BatchNormalization(), MaxPooling2D((2,2)),
    Conv2D(64,  (3,3), activation='relu', padding='same'),
    BatchNormalization(), MaxPooling2D((2,2)),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(), MaxPooling2D((2,2)),
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])
model_img.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, min_delta=0.005, restore_best_weights=True)
datagen = ImageDataGenerator(
    rotation_range=15, width_shift_range=0.1, height_shift_range=0.1,
    zoom_range=0.1, horizontal_flip=True
)
train_gen = datagen.flow(X_img_train, y_train, batch_size=64)

history_img = model_img.fit(
    train_gen,
    validation_data=(X_img_val, y_val),
    epochs=50,
    callbacks=[early_stop]
)

# ── Fusión con SVM ────────────────────────────────────────────────────────────
# Obtenemos probabilidades sobre TRAIN+VAL para entrenar el SVM
pred_img_train = model_img.predict(np.concatenate([X_img_train, X_img_val]))
pred_tab_train = model_tab.predict(np.concatenate([X_tab_train, X_tab_val]))
y_train_svm    = np.argmax(np.concatenate([y_train, y_val]), axis=1)

pred_img_test = model_img.predict(X_img_test)
pred_tab_test = model_tab.predict(X_tab_test)

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

fusion_train = np.concatenate([pred_img_train, pred_tab_train], axis=1)
fusion_test  = np.concatenate([pred_img_test,  pred_tab_test],  axis=1)
y_true = np.argmax(y_test, axis=1)

print("Entrenando SVM de fusión...")
clf = SVC(kernel='rbf', probability=True, random_state=42)
clf.fit(fusion_train, y_train_svm)

y_pred_classes = clf.predict(fusion_test)
acc = accuracy_score(y_true, y_pred_classes)
print(f"Accuracy Late Fusion (SVM): {acc:.4f}")

model_img.save('../models/late_fusion_image_model.h5')
model_tab.save('../models/late_fusion_tabular_model.h5')

plt.plot(history_img.history['accuracy'], label='Train Accuracy')
plt.plot(history_img.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy — Modelo Imágenes (Late Fusion)')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.show()

ValueError: Found input variables with inconsistent numbers of samples: [7010, 3005]

: 

## Evaluación detallada por clase

El accuracy global no es suficiente en diagnóstico médico. Aquí vemos cuánto acierta el modelo en **cada tipo de lesión** por separado, y lo comparamos con el **baseline ZeroR** (lo que conseguiría un modelo que predice siempre la clase más frecuente, sin aprender nada).

In [ ]:
from sklearn.metrics import classification_report
from collections import Counter

# Baseline ZeroR
most_common = Counter(y_true).most_common(1)[0][1]
baseline = most_common / len(y_true)
print(f'Baseline ZeroR (predecir siempre la clase más frecuente): {baseline:.2%}')
print(f'Accuracy Late Fusion (SVM): {acc:.2%}')
print(f'Mejora sobre baseline: {acc - baseline:+.2%}')

print('Resultados por clase diagnóstica:')
print(classification_report(y_true, y_pred_classes, target_names=le.classes_))

## Curvas ROC y matriz de confusión

La **curva ROC** mide cómo de bien el modelo separa cada clase del resto. El **AUC** (área bajo la curva) resume esto en un número: 1.0 = perfecto, 0.5 = aleatorio (no aprende nada).

En diagnóstico médico el AUC del **melanoma** (`mel`) es especialmente crítico: un falso negativo (diagnosticar como benigno algo maligno) tiene consecuencias graves.

La **matriz de confusión** muestra dónde se equivoca el modelo: qué clases confunde entre sí.

In [ ]:
import joblib, os
from sklearn.metrics import roc_curve, auc, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import label_binarize

# ── Guardar SVM (necesario para notebook 07) ──────────────────────────────────
os.makedirs('../models', exist_ok=True)
joblib.dump(clf, '../models/late_fusion_svm.pkl')
print("SVM guardado en ../models/late_fusion_svm.pkl")

# ── Curvas ROC ────────────────────────────────────────────────────────────────
y_pred_probs_svm = clf.predict_proba(fusion_test)
y_true_bin = label_binarize(y_true, classes=range(len(le.classes_)))

fig, ax = plt.subplots(figsize=(9, 6))
for i, (clase, color) in enumerate(zip(le.classes_, plt.cm.tab10.colors)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_probs_svm[:, i])
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{clase} (AUC = {auc(fpr, tpr):.2f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Aleatorio (AUC = 0.50)')
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Sensibilidad (Recall / TPR)')
ax.set_title('Curvas ROC por clase — Late Fusion (SVM)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

macro_auc = roc_auc_score(y_true_bin, y_pred_probs_svm, multi_class='ovr', average='macro')
print(f'AUC macro (promedio One-vs-Rest): {macro_auc:.4f}')

# ── Matriz de confusión ───────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred_classes)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(
    ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matriz de confusión — Late Fusion (SVM)')
plt.tight_layout()
plt.show()

# Modelo Late fusion: Conclusiones
En este modelo hemos alcanzado un valor de 0.75 aprox. el cual no es malo, pero es peor que en nuestro modelo Early fusion. 
Vemos que esto sucede porque, al procesar las dos modalidades de forma independiente hasta el final, la red no puede aprender interacciones profundas entre modalidades (es decir, yo he configurado que el peso de la parte tabular sea 0.4, y el peso de la parte de imágenes sea 0.6, pero no es una manera muy compleja de hacerlo)
Además, el aprendizaje para clasificar ambas modalidades no se hace de manera conjunta. 
Este tipo de red se utiliza en casos donde se busca la modularidad y la robustez, dado que al no tener interacciones tan complejas entre las dimensiones, es fácil sustituir una variable por otra
